In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score
from PIL import Image
import timm

# --- CONFIG ---
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
IMG_SIZE = 224
BATCH_SIZE = 32
LR = 1e-4

# --- METRICS ---
def get_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(5))
    sens, spec = [], []
    for i in range(5):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fp + fn)
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    return np.array(sens), np.array(spec)

# --- DATA ---
class AptosDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# --- ARCHITECTURE: ResNet-RS (Revised Scaling) ---
# This is highly stable for CUDA 11/12 and T4 GPUs
class StableAptosNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('resnetrs50', pretrained=True, num_classes=0)
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 5)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

# --- TRAINING ---
def run_experiment():
    # Force usage of ONLY the first GPU to prevent multi-GPU sync errors
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Deployment Device: {device}")
    
    model = StableAptosNet().to(device)
    
    train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_1.csv'))
    counts = train_df.iloc[:, 1].value_counts()
    class_weights = {i: 1.0/counts[i] for i in range(5)}
    class_weights[3] *= 4.0 # Maximum focus on Severe
    class_weights[4] *= 2.0 
    
    weights = [class_weights[c] for c in train_df.iloc[:, 1]]
    sampler = WeightedRandomSampler(weights, len(weights))

    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)
    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_kappa = 0
    for epoch in range(15):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                preds.extend(torch.argmax(out, 1).cpu().numpy())
                targets.extend(labels.cpu().numpy())

        kappa = cohen_kappa_score(targets, preds, weights='quadratic')
        print(f"Epoch {epoch+1} | Kappa: {kappa:.4f}")
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(model.state_dict(), 'stable_model.pth')

    # FINAL DOCS
    print("\n" + "="*50)
    print("      FINAL CLINICAL PERFORMANCE REPORT")
    print("="*50)
    model.load_state_dict(torch.load('stable_model.pth'))
    model.eval()
    
    f_preds, f_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            f_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
            f_targets.extend(labels.cpu().numpy())

    sens, spec = get_metrics(f_targets, f_preds)
    f1 = f1_score(f_targets, f_preds, average=None)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    
    for i in range(5):
        print(f"{classes[i]:<15} | Sensitivity: {sens[i]:.4f} | Specificity: {spec[i]:.4f} | F1: {f1[i]:.4f}")

    print("-" * 50)
    print(f"Final Quadratic Kappa: {cohen_kappa_score(f_targets, f_preds, weights='quadratic'):.4f}")

if __name__ == "__main__":
    run_experiment()

Deployment Device: cuda


model.safetensors:   0%|          | 0.00/143M [00:00<?, ?B/s]

Epoch 1 | Kappa: 0.8088
Epoch 2 | Kappa: 0.8091
Epoch 3 | Kappa: 0.8633
Epoch 4 | Kappa: 0.8884
Epoch 5 | Kappa: 0.8438
Epoch 6 | Kappa: 0.8737
Epoch 7 | Kappa: 0.8628
Epoch 8 | Kappa: 0.8846
Epoch 9 | Kappa: 0.8915
Epoch 10 | Kappa: 0.8832
Epoch 11 | Kappa: 0.8939
Epoch 12 | Kappa: 0.8745
Epoch 13 | Kappa: 0.8953
Epoch 14 | Kappa: 0.8762
Epoch 15 | Kappa: 0.8905

      FINAL CLINICAL PERFORMANCE REPORT
No DR           | Sensitivity: 0.9942 | Specificity: 0.9794 | F1: 0.9856
Mild            | Sensitivity: 0.5000 | Specificity: 0.9724 | F1: 0.5797
Moderate        | Sensitivity: 0.8846 | Specificity: 0.8664 | F1: 0.7965
Severe          | Sensitivity: 0.1364 | Specificity: 0.9826 | F1: 0.1935
Proliferative   | Sensitivity: 0.6071 | Specificity: 0.9734 | F1: 0.6296
--------------------------------------------------
Final Quadratic Kappa: 0.8925
